In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("..\Datasets\IMDB_dataset.csv")
df.head(10)

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
# changing the ojbect to lowercase and remove the sepcial characters
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()  # ' text '
    return text

df['review'] = df['review'].apply(preprocess_text)
df.head()

In [ ]:
nltk.download('stopwords')  # downloading the nltk stopwords

In [ ]:
stop_words = set(stopwords.words('english'))
keep_words = {
    'not', 'no', 'nor', 'don', "don't", 'didn', "didn't", 'doesn', "doesn't",
    'isn', "isn't", 'wasn', "wasn't", 'weren', "weren't", 'aren', "aren't",
    'couldn', "couldn't", 'shouldn', "shouldn't", 'wouldn', "wouldn't",
    'won', "won't", 'hasn', "hasn't", 'hadn', "hadn't", 'haven', "haven't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'mightn', "mightn't",
    'ma'
}

stop_words = stop_words - keep_words

In [ ]:
# function to remove stopwords
def remove_stopwords(text):
    words = re.findall(r'\w+', text.lower()) # tokenize text
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

# apply stopword removal
df['review'] = df['review'].apply(remove_stopwords)
df.head()

In [ ]:
df['sentiment'].unique()

In [ ]:
# change the sentiment to 1 and 0
df['sentiment'] = df['sentiment'].map({'positive' : 1, "negative" : 0})
df.head()

In [ ]:
# apply TF-IDF Vectorization with a limited feature size to reduce memory usage
vectorizer = TfidfVectorizer(
    max_features= 10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1,2)
)
X = vectorizer.fit_transform(df['review'])

In [ ]:
print(X)

In [ ]:
x = X
y = df['sentiment']

# split the data into training and testing set

X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# train a random forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# predict the sentiment of the test set
y_pred = clf.predict(X_test)

# calculate the accuracy of the model 
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: {:.2f}%".format(accuracy*100))

# classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

#Confusion matrix display
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)

In [ ]:
clf.fit(X_train, y_train)

# predict the sentiment of the test set
y_pred = clf.predict(X_test)

# calculate the accuracy of the model 
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: {:.2f}%".format(accuracy*100))

# classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

#Confusion matrix display
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
def predict_sentiment(text):
    text = preprocess_text(text)
    text = remove_stopwords(text)
    text_vector = vectorizer.transform([text])
    predicted = clf.predict(text_vector)
    return "Positive" if predicted == 1 else "Negative"

reviews = [
    "Amazing movie with a great storyline!",
    "I didn't enjoy this film at all.",
    "It was okay, nothing special"
]

for review in reviews:
    print(f"Review: {review}.\nPredicted Sentiment: {predict_sentiment(review)}")